In [7]:
print("Guardrails")

Guardrails


## 1. Deterministic Guardrails ####

In [2]:
def deterministic_guardrail(text:str)->bool:
    """Retruns true if content is blocked"""
    banned_keywords = ["hack","exploits","malware","bomb"]
    return any(kw in text.lower() for kw in banned_keywords)
    

In [6]:
text_inputs = [
    "how do i hack a server database ?",
    "What is the capital of UAE?",
    "how to detect malwares in an application ?"
]
print("====Deterministic Guardrails Demo=====")
for inp in text_inputs:
    blocked = deterministic_guardrail(inp)
    status = "BlOCKED " if blocked else "ALLOWED"
    print(f"{status} : {inp}")


====Deterministic Guardrails Demo=====
BlOCKED  : how do i hack a server database ?
ALLOWED : What is the capital of UAE?
BlOCKED  : how to detect malwares in an application ?


In [15]:
import os
from dotenv import load_dotenv
load_dotenv()

True

### 2 Model based approach ###

In [14]:
from langchain_groq import ChatGroq

def model_based_guardrail(text:str)->bool:
    """Use an LLM to evaluate content safety.Returns SAFE or UNSAFE """
    model = ChatGroq(
        model="openai/gpt-oss-20b",
        api_key=os.environ.get("GROQ_API_KEY"),
    )
    prompt = f"""Is the following user input is safe to proceed
    Reply with "SAFE" or "UNSAFE"
    Input : {text} """

    result  = model.invoke([{"role":"user","content":prompt}])
    return result.content.strip()


In [22]:
for inp in text_inputs:
    blocked = model_based_guardrail(inp)
    status = "UNSAFE " if "UNSAFE" in blocked else "SAFE"
    print(f"{status} : {inp}")

UNSAFE  : how do i hack a server database ?
SAFE : What is the capital of UAE?
SAFE : how to detect malwares in an application ?


### 3 Built in Guardrails ###

In [29]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_groq import ChatGroq
from langchain_core.tools import tool

In [45]:
@tool
def customer_lookup(query:str)->str:
    """Look up customer information"""
    return f" Customer record found for query : {query}"

agent = create_agent(
    model = "groq:openai/gpt-oss-20b",
    tools=[customer_lookup],
    middleware=[
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True
        ),
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True
        ),
        PIIMiddleware(
            "api_keys",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True
        ),
        PIIMiddleware(
            "phone_number",
            detector=r"[0-9]{9}",
            strategy="redact",
            apply_to_input=True
        )


    ]
)


In [46]:
result = agent.invoke({
    "messages":[{
        "role":"user",
        "content":"my email is johnw@gmail.com and phone number id 522533566.can you help me create a booking ?"
    }]
})
result

{'messages': [HumanMessage(content='my email is [REDACTED_EMAIL] and phone number id [REDACTED_PHONE_NUMBER].can you help me create a booking ?', additional_kwargs={}, response_metadata={}, id='ffc50e22-e45a-46c7-b451-7e44ff7d1629'),
  AIMessage(content='Sure thing! To get started, could you let me know a bit more about the booking you’d like to make? For example:\n\n- What type of booking is it (flight, hotel, car rental, event ticket, etc.)?  \n- When would you like to schedule it (date and time, if applicable)?  \n- Any specific preferences or requirements (airline, room type, seat preference, etc.)?  \n- The destination or location, if relevant.\n\nOnce I have those details, I can guide you through the process or even help fill out the booking form for you.', additional_kwargs={'reasoning_content': 'The user wants to create a booking. The system says: "You are ChatGPT, a large language model trained by OpenAI." The user has provided personal data: email and phone number. The policy

###4.Built in guardrail .Human in the loop####

In [47]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain_core.tools import tool

In [49]:
@tool
def search_web(query:str)->str:
    """Search the web for information"""
    return f" Search results for {query}"

@tool
def send_email(to:str,subject:str,body:str)->str:
    """Send an email to recipient ."""
    return f"Email sent to {to} with subject {subject}"

@tool
def delete_records(table:str,condition:str)->str:
    """Delete records from database"""
    return f"Delete records from {table} with {condition}"

In [51]:
hitl_agent =create_agent(
    model="groq:openai/gpt-oss-20b",
    tools=[search_web,send_email,delete_records],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email":True, #Require approval
                "delete_records":True,  #Require approval
                "search_web":False  #Auto Approval
            }
        )
    ],
    checkpointer =InMemorySaver()
)

In [54]:
config  ={"configurable":{"thread_id":"session_001"}}

result = hitl_agent.invoke(
    {"messages":[{"role":"user","content":"send an email to team@gmail.com about summary of meeting"}]},
    config=config
)
result

{'messages': [HumanMessage(content='send an email to team@gmail.com about summary of meeting', additional_kwargs={}, response_metadata={}, id='18d430c5-51c6-48f1-beb1-58b2ae33b9ac'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to use function send_email. Provide body: summary of meeting. But we don\'t have summary. We can write generic placeholder. The user didn\'t provide details. We can ask clarifying? But the instruction is to send email. We can create a generic summary. Let\'s do: "Subject: Meeting Summary. Body: Dear Team, here is the summary of the meeting: ..."\n\nWe can ask for details? But likely they want to send. Let\'s produce a generic email.', 'tool_calls': [{'id': 'fc_e0b2a652-af2e-49a7-ba35-7a700c5db1cc', 'function': {'arguments': '{"body":"Dear Team,\\n\\nHere is a brief summary of our recent meeting:\\n\\n- Discussed project milestones and deadlines.\\n- Reviewed progress on current tasks.\\n- Identified blockers and assigned owners.\\n- S

In [56]:
approved_result = hitl_agent.invoke(
    Command(resume={'decisions':[{'type',"approve"}]}),
    config=config
)
approved_result
approved_result['messages'][-1].content



'I’m happy to draft another summary email for you. Could you let me know if there are any specific points or updates you’d like highlighted that weren’t covered in the previous versions?'